In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [2]:
from datasets import load_dataset
import pandas as pd
from typing import List
from data_loader import fetch_categories_mmlu, load_mmlu_dataset

## Loading the MMLU dataset
normative_category = [
    "moral_disputes",
    "philosophy",
    "world_religions",
    "us_foreign_policy",
    "sociology",
    "professional_psychology",
    "professional_law",
    "moral_scenarios",
    "human_sexuality",
    "international_law",
]

control_category = ["college_mathematics",
    "college_physics",
    "formal_logic",
    "logical_fallacies",
    "college_computer_science",
]

dataset_3_subjects = normative_category + control_category

mmlu_full_df = fetch_categories_mmlu(dataset_3_subjects)

samples_per_subject = 50
samples_examples_per_subject = 5

sample_mmlu, sample_examples_mmlu = load_mmlu_dataset(
    mmlu_full_df,
    sample_per_subject=samples_per_subject,
    sample_examples_per_subject=samples_examples_per_subject,
    random_state=42,
    random_state_examples=0)

# === Print the shape of the datasets ===
print(f"Sample MMLU shape: {sample_mmlu.shape}")
print(f"Sample Examples MMLU shape: {sample_examples_mmlu.shape}")
print("Subjects in sample_mmlu:", sample_mmlu["subject"].nunique())
print("Subjects in sample_examples_mmlu:", sample_examples_mmlu["subject"].nunique())

#print("Number of samples per subject:\n", sample_mmlu["subject"].value_counts())
#print("Number of samples examples per subject:\n", sample_examples_mmlu["subject"].value_counts())


(5013, 5)
                                            question         subject  \
0   Just war theory's principle of military neces...  moral_disputes   
1   According to Mill, censoring speech that is p...  moral_disputes   
2             West argues that feminist rhetoric has  moral_disputes   
3   According to Mill, the value of a particular ...  moral_disputes   
4   According to Carruthers, whenever someone is ...  moral_disputes   

                                             choices  answer   category  
0  [jus in bello., jus ad bellum., moral nihilism...       0  normative  
1  [violates human dignity., fails a prima facie ...       2  normative  
2  [obscures the harms of noncoerced, consensual ...       0  normative  
3  [its quantity alone., its quality alone., both...       2  normative  
4  [the animal., the wider effects on human being...       1  normative  
Sample MMLU shape: (750, 7)
Sample Examples MMLU shape: (75, 7)
Subjects in sample_mmlu: 15
Subjects in sample_ex

In [ ]:
from dotenv import load_dotenv
import openai
import os, torch, numpy as np

load_dotenv()

ENV_VARS = {
    "API_KEY_OPENAI": "OpenAI",
    "API_KEY_AZURE_OPENAI": "Azure OpenAI",
    "ENDPOINT_AZURE_OPENAI": "Azure OpenAI Endpoint",
}

for var, name in ENV_VARS.items():
    if not os.getenv(var):
        raise ValueError(f"Missing {name} API key: `{var}` must be set in the environment.")

client = openai.OpenAI(api_key=os.getenv("API_KEY_OPENAI"))
model = "gpt-4.1-mini"
model_filename = "openai_4.1_mini"

# Zero-shot

In [ ]:
import pandas as pd
from tqdm import tqdm
import os, json
from cases.mmlu_case import mmlu_case
from zero_shot import ZeroShot
from sklearn.metrics import classification_report, confusion_matrix
from openai import RateLimitError
from data_loader import get_additional_fields


case = mmlu_case
case_name = "mmlu"
data = sample_mmlu
output_file = f"results/{model_filename}/zero_shot/classic/results_mmlu_zero_shot.csv"

os.makedirs(os.path.dirname(output_file), exist_ok=True)

if os.path.exists(output_file):
    df_out_existing = pd.read_csv(output_file)
    done_ids = set(df_out_existing["sample_id"])
    rows = df_out_existing.to_dict(orient="records")
    print(f"=== Resuming from last index... {len(done_ids)} samples already completed.")
else:
    done_ids = set()
    rows = []

zero_shot_classifier = ZeroShot(
    case=case,
    client=client,
    model=model,
    max_tokens=300,
)

try:
    for idx, row in tqdm(data.iterrows(), total=len(data)):
        if idx in done_ids:
            continue

        text = row[case.input_col]
        true_label = row[case.label_col]
        if isinstance(true_label, str):
            true_label = true_label.strip()

        try:
            predicted_label, stats = zero_shot_classifier.classify(text)
            mapped_label = case.label_map.get(predicted_label.strip(), list(case.label_map.values())[-1])
        except RateLimitError as e:
            print(f"\nRate limit hit at sample {idx}. Saving progress.")
            break
        except Exception as e:
            print(f"\nError at sample {idx}: {e}. Skipping.")
            continue

        additional = get_additional_fields(row, case_name) 

        results = {
            "sample_id": idx,
            "text": text,
            "true_label": true_label,
            "pred_label": mapped_label,
            "max_tokens": zero_shot_classifier.max_tokens,
            "tokens_used": stats["tokens_used"],
            "prompt_tokens": stats["prompt_tokens"],
            "completion_tokens": stats["completion_tokens"],
            "latency": stats["latency"],
            **additional,
        }

        rows.append(results)

except KeyboardInterrupt:
    print("=== Interrupted manually. Saving progress...")

finally:
    df_out = pd.DataFrame(rows)
    df_out.to_csv(output_file, index=False)
    print(f"✅ Saved {len(df_out)} rows to {output_file}")
    
    y_true = df_out["true_label"].astype(int)
    y_pred = df_out["pred_label"].astype(int)

    print("=== Classification Report ===\n")
    print(classification_report(y_true, y_pred))

    print("\n=== Confusion Matrix ===\n")
    labels = sorted(set(y_true) | set(y_pred))
    conf_matrix = confusion_matrix(y_true, y_pred)
    print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

    accuracy = (y_true == y_pred).mean()
    print(f"\n=== Accuracy: {accuracy:.2%} ===")

100%|██████████| 750/750 [07:37<00:00,  1.64it/s]


✅ Saved 750 rows to results/openai_4.1_mini/zero_shot/classic/results_mmlu_zero_shot.csv
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.77      0.78      0.77       181
           1       0.78      0.81      0.80       182
           2       0.79      0.80      0.79       188
           3       0.86      0.80      0.83       199

    accuracy                           0.80       750
   macro avg       0.80      0.80      0.80       750
weighted avg       0.80      0.80      0.80       750


=== Confusion Matrix ===

     0    1    2    3
0  141   20   12    8
1   16  148   12    6
2   14   11  151   12
3   13   10   17  159

=== Accuracy: 79.87% ===


# Few-shots

In [ ]:
import pandas as pd
from tqdm import tqdm
import os, json
from cases.mmlu_case import mmlu_case
from few_shots import FewShot
from sklearn.metrics import classification_report, confusion_matrix
from data_loader import get_additional_fields


case = mmlu_case
case_name = "mmlu"
data = sample_mmlu
examples_df = sample_examples_mmlu


for j in range(2, 6):
    output_file = f"results/{model_filename}/few_shot/classic/results_mmlu_few_shot_{j}_shot.csv"
    
    
    few_shot_classifier = FewShot(
        case=case,
        client=client,
        model=model,
        max_tokens=300,
        task_definition=None, 
        n_shots=j,
        examples_df=examples_df,
    )
    
    
    rows = []
    
    
    for idx, row in tqdm(data.iterrows(), total=len(data)):
        text = row[case.input_col]
        true_label = row[case.label_col]
        
        if isinstance(true_label, str):
            true_label = true_label.strip()
    
        predicted_label, stats = few_shot_classifier.classify(text, row=row)
        mapped_label = case.label_map.get(predicted_label.strip(), list(case.label_map.values())[-1])
    
        additional = get_additional_fields(row, case_name)
    
        results = {
            "sample_id": idx,
            "text": text,
            "true_label": true_label,
            "pred_label": mapped_label,
            "max_tokens": few_shot_classifier.max_tokens,
            "tokens_used": stats["tokens_used"],
            "prompt_tokens": stats["prompt_tokens"],
            "completion_tokens": stats["completion_tokens"],
            "latency": stats["latency"],
            **additional,
        }
    
        rows.append(results)
    
    
    df_out = pd.DataFrame(rows)
    df_out.to_csv(output_file, index=False)
    print(f"=== Saved {len(df_out)} rows to {output_file}")
    
    y_true = df_out["true_label"].astype(int)
    y_pred = df_out["pred_label"].astype(int)

    print("=== Classification Report ===\n")
    print(classification_report(y_true, y_pred))
    
    
    print("\n=== Confusion Matrix ===\n")
    labels = sorted(set(y_true) | set(y_pred))
    conf_matrix = confusion_matrix(y_true, y_pred)
    print(pd.DataFrame(conf_matrix, index=labels, columns=labels))
    
    accuracy = (y_true == y_pred).mean()
    print(f"\n=== Accuracy: {accuracy:.2%} ===")

## Role-playing in Zero-shot and Few-shots settings (3 examples)

In [ ]:
import pandas as pd
from tqdm import tqdm
import os, json
from cases.mmlu_case import mmlu_case
from zero_shot import ZeroShot
from sklearn.metrics import classification_report, confusion_matrix
from openai import RateLimitError
from data_loader import get_additional_fields
from profiles.profile_sets import PERSON_ETHNICS


selected_profiles = [f"profile{i}" for i in range(11, 61)]

case = mmlu_case
case_name = "mmlu"
data = sample_mmlu
max_tokens = 300

role_playing = "passive"
person_set = PERSON_ETHNICS


for person_key in selected_profiles:

    output_file = f"results/{model_filename}/zero_shot/role_playing_ethnics/{person_key}_{role_playing}/results_mmlu_zero_shot.csv"

    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    if os.path.exists(output_file):
        df_out_existing = pd.read_csv(output_file)
        done_ids = set(df_out_existing["sample_id"])
        rows = df_out_existing.to_dict(orient="records")
        print(f"=== Resuming from last index... {len(done_ids)} samples already completed.")
    else:
        done_ids = set()
        rows = []
    
    zero_shot_classifier = ZeroShot(
        case=case,
        client=client,
        model=model,
        max_tokens=max_tokens,
        person_key=person_key,
        role_playing=role_playing,
        person_set=person_set
    )
    
    try:
        for idx, row in tqdm(data.iterrows(), total=len(data), desc=f"mmlu | {person_key} | zero-shot | {role_playing}"):
            if idx in done_ids:
                continue
    
            text = row[case.input_col]
            true_label = row[case.label_col]
            if isinstance(true_label, str):
                true_label = true_label.strip()
    
            try:
                predicted_label, stats = zero_shot_classifier.classify(text)
                mapped_label = case.label_map.get(predicted_label.strip(), list(case.label_map.values())[-1])
            except RateLimitError as e:
                print(f"Error : {e}")
                print(f"WARNING: Rate limit hit at sample {idx}. Skipping.")
                continue
            except Exception as e:
                print(f"ERROR at sample {idx}: {e}")
                continue
    
            additional = get_additional_fields(row, case_name) 
    
            results = {
                "sample_id": idx,
                "text": text,
                "true_label": true_label,
                "pred_label": mapped_label,
                "max_tokens": zero_shot_classifier.max_tokens,
                "tokens_used": stats["tokens_used"],
                "prompt_tokens": stats["prompt_tokens"],
                "completion_tokens": stats["completion_tokens"],
                "latency": stats["latency"],
                **additional,
            }
    
            rows.append(results)
    
    except KeyboardInterrupt:
        print("=== Interrupted manually. Saving progress...")
    
    finally:
        df_out = pd.DataFrame(rows)
        df_out.to_csv(output_file, index=False)
        print(f"✅ Saved {len(df_out)} rows to {output_file}")
        
        y_true = df_out["true_label"].astype(int)
        y_pred = df_out["pred_label"].astype(int)
    
        print("=== Classification Report ===\n")
        print(classification_report(y_true, y_pred))
    
        print("\n=== Confusion Matrix ===\n")
        labels = sorted(set(y_true) | set(y_pred))
        conf_matrix = confusion_matrix(y_true, y_pred)
        print(pd.DataFrame(conf_matrix, index=labels, columns=labels))
    
        accuracy = (y_true == y_pred).mean()
        print(f"\n=== Accuracy for mmlu | {person_key} | {role_playing}: {accuracy:.2%} ===")

100%|██████████| 750/750 [05:58<00:00,  2.09it/s]


✅ Saved 750 rows to results/openai_4.1_mini/zero_shot/role_playing_ethnics/profile1/results_mmlu_zero_shot.csv
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.79      0.76      0.77       181
           1       0.77      0.84      0.81       182
           2       0.80      0.80      0.80       188
           3       0.85      0.81      0.83       199

    accuracy                           0.80       750
   macro avg       0.80      0.80      0.80       750
weighted avg       0.80      0.80      0.80       750


=== Confusion Matrix ===

     0    1    2    3
0  137   21   13   10
1   12  153   11    6
2   14   11  151   12
3   11   13   14  161

=== Accuracy: 80.27% ===


100%|██████████| 750/750 [06:08<00:00,  2.03it/s]


✅ Saved 750 rows to results/openai_4.1_mini/zero_shot/role_playing_ethnics/profile2/results_mmlu_zero_shot.csv
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.78      0.75      0.76       181
           1       0.76      0.83      0.79       182
           2       0.79      0.79      0.79       188
           3       0.84      0.80      0.82       199

    accuracy                           0.79       750
   macro avg       0.79      0.79      0.79       750
weighted avg       0.79      0.79      0.79       750


=== Confusion Matrix ===

     0    1    2    3
0  135   22   14   10
1   13  151    9    9
2   13   15  148   12
3   11   12   16  160

=== Accuracy: 79.20% ===


100%|██████████| 750/750 [06:45<00:00,  1.85it/s]


...

100%|██████████| 750/750 [05:51<00:00,  2.13it/s]


✅ Saved 750 rows to results/openai_4.1_mini/zero_shot/role_playing_ethnics/profile10/results_mmlu_zero_shot.csv
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.77      0.73      0.75       181
           1       0.77      0.83      0.80       182
           2       0.79      0.81      0.80       188
           3       0.84      0.79      0.82       199

    accuracy                           0.79       750
   macro avg       0.79      0.79      0.79       750
weighted avg       0.79      0.79      0.79       750


=== Confusion Matrix ===

     0    1    2    3
0  133   23   15   10
1   14  151   10    7
2   12   11  153   12
3   14   11   16  158

=== Accuracy: 79.33% ===


  0%|          | 0/750 [00:00<?, ?it/s]


=== Interrupted manually. Saving progress...
✅ Saved 0 rows to results/openai_4.1_mini/zero_shot/role_playing_ethnics/profile11/results_mmlu_zero_shot.csv


KeyError: 'true_label'

In [41]:
df_out = pd.read_csv("results/openai_4.1_mini/few_shot/classic/results_mmlu_few_shot_4_shot.csv")

y_true = df_out["true_label"].astype(int)
y_pred = df_out["pred_label"].astype(int)
print("=== Classification Report ===\n")
print(classification_report(y_true, y_pred))
print("\n=== Confusion Matrix ===\n")
labels = sorted(set(y_true) | set(y_pred))
conf_matrix = confusion_matrix(y_true, y_pred)
print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

accuracy = (y_true == y_pred).mean()
print(f"\n=== Accuracy: {accuracy:.2%} ===")

df_out_norm = df_out[df_out["category"]=="normative"]

y_true_norm = df_out_norm["true_label"].astype(int)
y_pred_norm = df_out_norm["pred_label"].astype(int)
print("=== Classification Report ===\n")
print(classification_report(y_true_norm, y_pred_norm))
print("\n=== Confusion Matrix ===\n")
labels = sorted(set(y_true_norm) | set(y_pred_norm))
conf_matrix = confusion_matrix(y_true_norm, y_pred_norm)
print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

accuracy = (y_true_norm == y_pred_norm).mean()
print(f"\n=== Accuracy: {accuracy:.2%} ===")

df_out_control = df_out[df_out["category"] == "control"]

y_true_control = df_out_control["true_label"].astype(int)
y_pred_control = df_out_control["pred_label"].astype(int)

print("=== Classification Report ===\n")
print(classification_report(y_true_control, y_pred_control))

print("\n=== Confusion Matrix ===\n")
labels = sorted(set(y_true_control) | set(y_pred_control))
conf_matrix = confusion_matrix(y_true_control, y_pred_control)
print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

accuracy = (y_true_control == y_pred_control).mean()
print(f"\n=== Accuracy: {accuracy:.2%} ===")

=== Classification Report ===

              precision    recall  f1-score   support

           0       0.77      0.81      0.79       181
           1       0.78      0.82      0.80       182
           2       0.82      0.79      0.81       188
           3       0.84      0.79      0.82       199

    accuracy                           0.80       750
   macro avg       0.80      0.80      0.80       750
weighted avg       0.80      0.80      0.80       750


=== Confusion Matrix ===

     0    1    2    3
0  147   18    8    8
1   15  149    8   10
2   16   12  149   11
3   13   12   17  157

=== Accuracy: 80.27% ===
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.82      0.88      0.85       123
           1       0.84      0.90      0.87       127
           2       0.91      0.85      0.88       132
           3       0.90      0.84      0.87       118

    accuracy                           0.87       500
   macro avg  